# Day 30 — Week 4 Research Training Retrospective

**Theme:** From *reading a paper* to *owning a reproducible research pipeline*

This notebook closes Week 4. The goal is not to add more knowledge, but to make explicit what changed in the way I do research.

> Week 4 transition  
> **Paper Reading → Repository Understanding → Baseline Reproduction → Controlled Experiment → Error Analysis → Research Questions**

Primary case study: **DENSE — Dynamic Bundling with Large Language Models for Zero-Shot Inference on Text-Attributed Graphs (NeurIPS 2025)**.


## 1. Where I started

At the beginning of this week, my understanding of research reproduction was roughly:

```text
read paper
   ↓
clone repo
   ↓
run README command
   ↓
get a number
```

That view was incomplete.

A real reproduction required me to distinguish at least four layers:

```text
Paper claim
    ↓
Released implementation
    ↓
Executable / repaired implementation
    ↓
Controlled experimental evidence
```

A repository being public does **not** automatically mean that every README configuration is executable or that every implementation detail is complete.


## 2. The DENSE idea I now understand

The core DENSE pipeline can be reconstructed as:

```text
Text-Attributed Graph
        ↓
Bundle Sampling
        ↓
LLM queries bundled node texts
        ↓
Bundle-level pseudo label
        ↓
GNN optimization
        ↓
Bundle refinement
        ↓
Repeat optimization / refinement
        ↓
Zero-shot node prediction
```

The important conceptual shift is that the LLM is not simply classifying every node independently.

Instead:

$$
\text{LLM} \rightarrow \text{bundle-level supervision}
$$

and the GNN learns node predictions from this weak / aggregated supervision.

This connects the two topics I care about:

$$
\boxed{\text{LLM reasoning / supervision} + \text{graph structure}}
$$


## 3. What I learned from reading the repository

I stopped reading code from line 1 downward.

The useful execution-path reading strategy became:

```text
main()
  ↓
Solver.__init__()
  ↓
solve()
  ↓
bundle_presample()
  ↓
bundle_sample() / batch_bundle_query()
  ↓
bundle_optimize()
  ↓
bundle_resample()
  ↓
evaluate()
```

For each function I now ask:

1. What enters?
2. What shape does it have?
3. What semantic meaning does it carry?
4. What changes?
5. Where does the result go next?

This was much more effective than trying to understand every import and every line before knowing the program's control flow.


## 4. Tensor reasoning became concrete

One important example was:

```python
bundle_logits = logits[bundles_indices, :]
```

If

```text
logits          : [N, C]
bundles_indices : [B, K]
```

then advanced indexing produces:

```text
bundle_logits   : [B, K, C]
```

where:

- `B` = number of bundles
- `K` = nodes per bundle
- `C` = number of classes

So this tensor means:

> for every bundle, for every node in that bundle, what are the GNN logits over all classes?

Understanding the **meaning of dimensions**, rather than memorizing tensor syntax, made the later loss and refinement code much easier to understand.


## 5. Average loss vs. ranking loss

### Average supervision

The average branch computes:

$$
\bar{z}_B = \frac{1}{|B|}\sum_{v\in B} z_v
$$

and then:

$$
\mathcal{L}_{BE}
=
CE(\bar{z}_B, y_B)
$$

This does **not** mean that every node is individually forced to have label $y_B$.

Instead, the bundle as a whole is trained toward the LLM-provided bundle label.

### Ranking supervision

The ranking branch first computes node probabilities:

$$
p_v = \operatorname{softmax}(z_v)
$$

then averages them within a bundle:

$$
\bar{p}_B
=
\frac{1}{|B|}
\sum_{v\in B}p_v
$$

For the LLM label $y_B$:

$$
p_{\text{class}}=\bar{p}_B[y_B]
$$

and:

$$
p_{\max}=\max_c \bar{p}_B[c]
$$

The ranking penalty becomes positive when the LLM-provided class is not currently top-ranked.

So the practical structure is:

$$
\boxed{
\mathcal{L}
=
\mathcal{L}_{BE}
+
\mathcal{L}_{R}
}
$$

This explains why the `ranking` branch still contains the same cross-entropy component used by `average`.


## 6. Code review became part of reproduction

Reading the implementation exposed inconsistencies in the released reproduction path.

Examples encountered during this week included:

```text
README configuration
        ≠
implemented model / sampler support

function call
        ≠
function signature

ranking cross entropy
        missing target

solve()
        calls refinement
        but released refinement implementation was incomplete / absent
```

The key lesson is not merely that “the repo had bugs.”

The important lesson is:

> **A reproduction researcher must distinguish implementation defects from scientific claims.**

A runtime repair should be minimal and documented.  
A reconstructed algorithmic component should be marked as reconstructed.  
A new experimental modification should not be silently presented as part of the original method.


## 7. My repaired experimental pipeline

By the end of the week, the working local path became:

```text
Cora
 ↓
bundle sampling
 ↓
LLM bundle query
 ↓
bundle labels
 ↓
GCN / GIN
 ↓
average or ranking loss
 ↓
confidence-based refinement
 ↓
evaluation
```

I also learned to separate:

```text
Compatibility fix
Robustness fix
Algorithmic reconstruction
Experimental modification
```

This provenance distinction is essential if the work later becomes a report, research memo, or contribution to a real project.


## 8. The first controlled result

A controlled local experiment used the same basic configuration and changed the loss:

| Setting | Accuracy |
|---|---:|
| Average | 0.6845 |
| Ranking | 0.7399 |

Observed difference:

$$
0.7399 - 0.6845 = 0.0554
$$

or **+5.54 percentage points** in this run.

### What this supports

Under this specific local repaired configuration, ranking supervision produced a higher final accuracy than average supervision.

### What this does NOT establish

It does not establish that ranking is universally better because:

```text
repeat = 1
```

There is not yet a multi-seed estimate of:

$$
\mu \pm \sigma
$$

and the repaired local implementation is not identical to every configuration described in the released README.

This distinction between **observation** and **general conclusion** is one of the most important research habits from this week.


## 9. Refinement: from an abstract idea to a measurable mechanism

For an LLM-labeled bundle $B$ with label $y_B$, refinement examines:

$$
p(y=y_B\mid v)
$$

for each node $v$ and removes:

$$
v^*
=
\arg\min_{v\in B}
p(y=y_B\mid v)
$$

I instrumented this mechanism and observed real examples such as:

```text
LLM bundle label = 6

true labels:
6   6   2   6   6

GNN confidence for class 6:
.1973  .1897  .1736  .1874  .1844
               ↑
             remove

label consistency:
0.80 → 1.00
```

But I also observed cases where the lowest-confidence node was **not** the noisy node and consistency decreased.

That observation turns implementation understanding into a research question:

> Is confidence-based refinement actually identifying noisy nodes, or is part of its benefit simply caused by shrinking the bundle?


## 10. From debugging to research questions

This week produced several questions that can be tested rather than merely discussed.

```text
Question 1
Does ranking supervision consistently outperform average supervision?
→ multiple seeds

Question 2
Does confidence-based refinement outperform random removal?
→ controlled refinement ablation

Question 3
Does refinement increase bundle label consistency / purity?
→ before-vs-after diagnostics

Question 4
How sensitive is DENSE to GNN architecture?
→ GCN vs GIN under controlled configuration

Question 5
How do sampling strategies affect bundle quality?
→ neighbor / feature / reconstructed hybrid
```

This is the transition I wanted from Week 4:

```text
code problem
    ↓
mechanism
    ↓
hypothesis
    ↓
controlled experiment
```


## 11. GIN: restoring a missing reproduction capability

The released README specifies GIN for the Cora configuration, while the model factory I inspected did not originally expose it.

I added PyG's GIN through the same model interface and verified:

```text
synthetic forward test
        PASS
          ↓
real Cora pipeline
        PASS
          ↓
ranking loss
        PASS
          ↓
refinement using GIN confidence
        PASS
          ↓
evaluation
        PASS
```

This was a useful example of a disciplined implementation task:

> Do not rewrite an architecture unnecessarily. Restore the missing model through the framework's existing abstraction and verify interface compatibility first.

The full Cora GIN experiment is the next formal reproduction run.


## 12. Systems lessons were also research lessons

The LLM API pipeline exposed another class of problems:

```text
rate limits
Cloudflare / upstream failures
partial runs
missing persistent query results
long-running SSH jobs
```

I learned practical tools and habits:

```text
screen
  → keep long jobs alive

tee / logs
  → preserve experiment output

defensive API handling
  → don't crash on malformed / missing responses

cache / persistent artifacts
  → avoid paying for the same LLM supervision repeatedly

Git branches / commits
  → preserve known-good experimental states
```

These are not separate from research.

A result that cannot be recovered, traced, or reproduced is weak evidence even if the final number looks good.


## 13. The workflow I want to keep

My research workflow after Week 4 is:

```text
1. Read the paper for the research question
                  ↓
2. Reconstruct the method as a pipeline
                  ↓
3. Trace the repository execution path
                  ↓
4. Establish a minimal executable baseline
                  ↓
5. Separate fixes from algorithm changes
                  ↓
6. Reproduce one result
                  ↓
7. Change ONE variable
                  ↓
8. Record evidence
                  ↓
9. Explain the mechanism
                  ↓
10. Turn anomalies into hypotheses
```

The key change is:

> **I no longer treat “the code ran” as the end of reproduction.**

Running code is only the beginning of understanding why a method works.


## 14. Week 2 → Week 3 → Week 4

The last three weeks now form one coherent progression:

```text
Week 2 — Graph / GNN foundations
GCN → GraphSAGE → GAT
message passing / aggregation / inductive learning
                    │
                    ▼
Week 3 — LLM systems + agents + Graph × LLM
inference → ReAct → agent → Graph RAG / KG
                    │
                    ▼
Week 4 — Actual research workflow
paper → repo → reproduction → ablation → mechanism
```

This matters because DENSE sits exactly at the intersection:

```text
             Graph × LLM
                 │
        ┌────────┴────────┐
        │                 │
       LLM               GNN
 bundle supervision   graph learning
        │                 │
        └────────┬────────┘
                 ↓
         structured reasoning
```

So Week 4 was not a random paper reproduction. It connected the technical foundations from the previous weeks to an actual research artifact.


## 15. What I can now explain without hiding behind the paper

At the end of Week 4, I should be able to explain DENSE in my own words:

> DENSE reduces the unreliability and structural blindness of isolated LLM node queries by grouping nearby text-attributed nodes into bundles. An LLM assigns a label to each bundle, and these bundle-level labels provide weak supervision for a GNN. The GNN is optimized using bundle-level objectives, and the bundles are dynamically refined using the GNN's confidence so that potentially noisy members can be removed. The important research questions are therefore not only whether the final accuracy is high, but whether bundling improves supervision quality, whether the ranking objective adds useful pressure, and whether confidence-based refinement actually identifies noisy nodes.

If I can defend every sentence above by pointing to the paper, code, or my own controlled evidence, then I understand the project at a research level rather than only at a tutorial level.


## 16. Current evidence ledger

### Established locally

```text
✓ DENSE execution path understood
✓ released-code inconsistencies identified
✓ repaired GCN pipeline executable
✓ ranking vs average controlled run completed
✓ ranking: 0.7399
✓ average: 0.6845
✓ refinement confidence inspected at node level
✓ refinement can improve OR hurt label consistency
✓ GIN added to model factory
✓ GIN forward test passed
✓ GIN real-Cora smoke pipeline passed
```

### Still in progress / not yet established

```text
○ full Cora GIN 100-bundle result
○ multi-seed mean ± std
○ confidence refinement vs random removal
○ robust refinement statistics over all bundles
○ SAGE support
○ provenance-faithful hybrid sampler
○ broader multi-dataset reproduction
```

Keeping these two categories separate prevents unfinished work from silently becoming a “result.”


## 17. The next research step

The immediate next step is **not another course**.

It is to finish the experimental loop:

```text
Full Cora GIN baseline
        ↓
multiple seeds
        ↓
ranking vs average
        ↓
confidence vs random refinement
        ↓
analyze bundle-level diagnostics
        ↓
write a compact research memo
```

After that, extending model/sampling support (SAGE / hybrid / WikiCS) becomes meaningful because there is already a trustworthy baseline and experimental methodology.

The goal is no longer:

> “Can I run this repository?”

The goal becomes:

> **“Can I identify an assumption in the method, design a controlled test for it, interpret the evidence, and explain what should be tried next?”**


# Week 4 Final Takeaway

This week changed my unit of progress.

Before:

```text
paper read
code run
accuracy obtained
```

Now:

```text
Research progress
=
understanding
+ reproducibility
+ controlled evidence
+ mechanism analysis
+ new testable questions
```

The most valuable outcome is not the current accuracy number.

It is that I have started moving through the complete loop:

$$
\boxed{
\text{Paper}
\rightarrow
\text{Code}
\rightarrow
\text{Experiment}
\rightarrow
\text{Evidence}
\rightarrow
\text{Hypothesis}
}
$$

That is the workflow I want to carry into the next stage of research.
